<a href="https://colab.research.google.com/github/Kerry-lab/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Step 1: Identify the Task
Guide an autonomous cargo trailer from Nairobi to Kisumu using our Exercise 3 road graph and shortest-distance search, while adapting to road hazards and trailer physics.  
**Goal:** Deliver cargo safely and on time with zero crashes, jackknifing, or cargo spoilage.


### Step 2: Assemble Relevant Knowledge
The trailer agent needs:
- **Highway network & distances:** 17 towns and segment distances in km from Practical Exercise 3.
- **Terrain & road facts:** Mau Summit (~2,534 m) steep gradients, active roadworks/diversions, weather/visibility risks, and safe trailer actions.
- **APIs / Web Scraping:** None used.

### Step 3: Define the Vocabulary
- **Constants:** Towns (Nairobi, Naivasha, Nakuru, Mau Summit, Londiani, Muhoroni, Kisumu, etc.), Road States (Clear, Steep_Climb, Roadworks, Slippery, Blocked).
- **Predicates (Python keys & rules):** Route(start, goal), RoadStatus(segment), SafeAction(state).

### Step 4: Encode General Knowledge about the Domain
Store baseline road conditions along the optimal route and general safe-driving rules for a heavy trailer.

In [1]:
# General facts: known condition per highway segment
segment_status = {
    ("Nairobi", "Naivasha"): "Roadworks",
    ("Naivasha", "Nakuru"): "Clear",
    ("Nakuru", "Mau Summit"): "Steep_Climb",
    ("Mau Summit", "Londiani"): "Steep_Climb",
    ("Londiani", "Muhoroni"): "Slippery",
    ("Muhoroni", "Kisumu"): "Clear",
}

# General rules: safe action for each road condition
trailer_rules = {
    "Clear": "Cruise safely and keep safe trailer spacing.",
    "Steep_Climb": "Use low gear and engine braking; avoid sudden steering.",
    "Roadworks": "Slow down for diversions and watch for heavy machinery.",
    "Slippery": "Reduce speed early, brake gently, and prevent trailer sway.",
    "Blocked": "Stop safely and reroute via Uniform Cost Search."
}

print("Step 4 loaded:", len(segment_status), "segments and", len(trailer_rules), "rules.")

Step 4 loaded: 6 segments and 5 rules.


### Step 5: Add New Facts to the Knowledge Base
Dynamically add new road condition facts to the knowledge base (both in code and via user input) and inspect the updated state.

In [4]:
from string import capwords

def clean(name):
    """Normalise town names so they match the graph keys (e.g. Murang'a, Mau Summit)."""
    return capwords(name.strip())

def show_kb():
    print("\n--- Current Trailer Knowledge Base ---")
    print("[Segment Statuses]")
    for seg, status in segment_status.items():
        print(f"  {seg[0]} -> {seg[1]:12} : {status}")
    print("\n[Driving Rules]")
    for status, action in trailer_rules.items():
        print(f"  {status:12} -> {action}")
    print("------------------------------------")

def add_segment_fact(origin, destination, status):
    """Add or update a highway segment status directly in code."""
    segment_status[(clean(origin), clean(destination))] = status.strip()
    print(f"Added fact: ({clean(origin)}, {clean(destination)}) -> '{status.strip()}'")

def add_rule(status, action):
    """Add a new driving rule for a road status."""
    trailer_rules[status.strip()] = action.strip()
    print(f"Added rule: '{status.strip()}' -> '{action.strip()}'")

# 1) New facts added directly in code
add_segment_fact("Nakuru", "Eldoret", "Slippery")
add_segment_fact("Bomet", "Kericho", "Blocked")

# 2) New rule added directly in code
add_rule("Foggy", "Reduce speed, use fog lights, and increase following distance.")

def add_fact_interactive():
    """Add a new segment status via user/sensor input."""
    origin = clean(input("Enter start town: "))
    dest = clean(input("Enter next town: "))
    status = input(f"Enter road status for {origin} -> {dest}: ").strip()
    segment_status[(origin, dest)] = status
    print(f"New fact added: ({origin}, {dest}) -> '{status}'")

    # If the status has no rule yet, ask for one so the agent knows what to do
    if status not in trailer_rules:
        action = input(f"What should the trailer do when the road is {status}? ")
        add_rule(status, action)

# 3) New fact added through user input
add_fact_interactive()

# View updated KB
show_kb()

Added fact: (Nakuru, Eldoret) -> 'Slippery'
Added fact: (Bomet, Kericho) -> 'Blocked'
Added rule: 'Foggy' -> 'Reduce speed, use fog lights, and increase following distance.'
Enter start town: Kisii
Enter next town: Nairobi
Enter road status for Kisii -> Nairobi: Not bad
New fact added: (Kisii, Nairobi) -> 'Not bad'
What should the trailer do when the road is Not bad? Drive on normal speed
Added rule: 'Not bad' -> 'Drive on normal speed'

--- Current Trailer Knowledge Base ---
[Segment Statuses]
  Nairobi -> Naivasha     : Roadworks
  Naivasha -> Nakuru       : Clear
  Nakuru -> Mau Summit   : Steep_Climb
  Mau Summit -> Londiani     : Steep_Climb
  Londiani -> Muhoroni     : Slippery
  Muhoroni -> Kisumu       : Clear
  Nakuru -> Eldoret      : Slippery
  Bomet -> Kericho      : Blocked
  Nairobi -> Kisii        : good
  Kisii -> Nairobi      : Not bad

[Driving Rules]
  Clear        -> Cruise safely and keep safe trailer spacing.
  Steep_Climb  -> Use low gear and engine braking; avoi

### Step 6:Specific Problem Instance(What the trailer observes
 right now)**


In [7]:

start_city = "Nairobi"
goal_city = "Kisumu"

# Live observations from sensors/driver, kept separate from the general KB
live_observations = {}

def observe_road():
    """Record a current road condition reported by sensors."""
    town_a = clean(input("Trailer's current town: "))
    town_b = clean(input("Next town ahead: "))
    status = input("Observed road condition (e.g. Clear, Slippery, Blocked, Foggy): ").strip().title()
    live_observations[(town_a, town_b)] = status
    print(f"Observation recorded: ({town_a}, {town_b}) -> '{status}'")

print(f"Trip: {start_city} -> {goal_city}")
observe_road()

print("\n--- Current Observations ---")
for seg, status in live_observations.items():
    print(f"  {seg[0]} -> {seg[1]} : {status}")

Trip: Nairobi -> Kisumu
Trailer's current town: Mau Summit
Next town ahead: Londiani
Observed road condition (e.g. Clear, Slippery, Blocked, Foggy): Blocked
Observation recorded: (Mau Summit, Londiani) -> 'Blocked'

--- Current Observations ---
  Mau Summit -> Londiani : Blocked


### Step 7: Pose Queries and Get Answers

The agent queries the knowledge base to plan the Nairobi to Kisumu trip. It runs UCS for the shortest route (356 km), checks the KB for ***Blocked ***segments (general facts plus live observations), reroutes with UCS if needed, then looks up the driving rule for each segment on the final route.

In [8]:
import heapq

# Exercise 3 road graph (same as before)
highway_graph = {
    "Nairobi": {"Thika": 42, "Naivasha": 92, "Narok": 140},
    "Thika": {"Nairobi": 42, "Murang'a": 40},
    "Murang'a": {"Thika": 40, "Nyeri": 65},
    "Nyeri": {"Murang'a": 65, "Nyahururu": 70},
    "Naivasha": {"Nairobi": 92, "Nakuru": 64, "Nyahururu": 95},
    "Nyahururu": {"Nyeri": 70, "Naivasha": 95, "Nakuru": 90},
    "Nakuru": {"Naivasha": 64, "Nyahururu": 90, "Mau Summit": 56, "Eldoret": 155},
    "Narok": {"Nairobi": 140, "Bomet": 75},
    "Bomet": {"Narok": 75, "Kericho": 55, "Kisii": 70},
    "Mau Summit": {"Nakuru": 56, "Londiani": 24},
    "Londiani": {"Mau Summit": 24, "Muhoroni": 75, "Kericho": 45},
    "Kericho": {"Londiani": 45, "Bomet": 55, "Kisumu": 100},
    "Muhoroni": {"Londiani": 75, "Kisumu": 45},
    "Kisii": {"Bomet": 70, "Kisumu": 115},
    "Eldoret": {"Nakuru": 155, "Kakamega": 90},
    "Kakamega": {"Eldoret": 90, "Kisumu": 55},
    "Kisumu": {}
}

# ---------- Querying the knowledge base ----------
def get_status(a, b):
    """Current status of a segment. Live observations override general facts.
    Checks both directions because roads are two-way."""
    for source in (live_observations, segment_status):
        for key in ((a, b), (b, a)):
            if key in source:
                return source[key]
    return "Clear"  # no information -> assume clear

def blocked_segments():
    """All segments the KB says are Blocked (either KB source, both directions)."""
    blocked = set()
    for source in (segment_status, live_observations):
        for (a, b), status in source.items():
            if status == "Blocked":
                blocked.add((a, b))
                blocked.add((b, a))
    # a live 'Clear' report can reopen a road that the general KB marked Blocked
    for (a, b), status in live_observations.items():
        if status != "Blocked":
            blocked.discard((a, b))
            blocked.discard((b, a))
    return blocked

def action_for(status):
    return trailer_rules.get(status, "Unknown condition - slow down and proceed with caution.")

# ---------- UCS that respects the KB ----------
def uniform_cost_search(graph, start, goal, blocked=frozenset()):
    frontier = [(0, start, [start])]
    best = {start: 0}
    explored = set()

    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored or cost > best.get(node, float("inf")):
            continue
        if node == goal:
            return path, cost
        explored.add(node)

        for neighbor, weight in graph.get(node, {}).items():
            if (node, neighbor) in blocked or neighbor in explored:
                continue
            new_cost = cost + weight
            if new_cost < best.get(neighbor, float("inf")):
                best[neighbor] = new_cost
                heapq.heappush(frontier, (new_cost, neighbor, path + [neighbor]))
    return None, float("inf")

# ---------- Step 7: the agent plans and queries ----------
def plan_trip(start, goal):
    print("=" * 60)
    print(f"TRAILER AGENT: {start} -> {goal}")
    print("=" * 60)

    # 1) Baseline route ignoring the KB (the Exercise 3 answer)
    base_path, base_cost = uniform_cost_search(highway_graph, start, goal)
    print(f"Original optimal route : {' -> '.join(base_path)} ({base_cost} km)")

    # 2) Check the KB for blocked segments on that route
    blocked = blocked_segments()
    hit = [(a, b) for a, b in zip(base_path, base_path[1:]) if (a, b) in blocked]

    if hit:
        for a, b in hit:
            print(f"\nALERT: {a} -> {b} is Blocked")
            print(f"Rule: {action_for('Blocked')}")
        path, cost = uniform_cost_search(highway_graph, start, goal, blocked)
        if path is None:
            print("\nNo safe route available. Hold at the nearest safe stop.")
            return
        print(f"\nRerouted route : {' -> '.join(path)} ({cost} km)")
        print(f"Extra distance : +{cost - base_cost} km")
    else:
        path, cost = base_path, base_cost
        print("\nNo blocked segments on the route. Keeping original route.")

    # 3) Driving instructions for every segment on the final route
    print("\n--- Driving Instructions ---")
    for a, b in zip(path, path[1:]):
        status = get_status(a, b)
        print(f"{a} -> {b:12} [{status}]: {action_for(status)}")

    print(f"\nFinal route: {' -> '.join(path)} | Total: {cost} km")

plan_trip(start_city, goal_city)

TRAILER AGENT: Nairobi -> Kisumu
Original optimal route : Nairobi -> Naivasha -> Nakuru -> Mau Summit -> Londiani -> Muhoroni -> Kisumu (356 km)

ALERT: Mau Summit -> Londiani is Blocked
Rule: Stop safely and reroute via Uniform Cost Search.

Rerouted route : Nairobi -> Narok -> Bomet -> Kisii -> Kisumu (400 km)
Extra distance : +44 km

--- Driving Instructions ---
Nairobi -> Narok        [Clear]: Cruise safely and keep safe trailer spacing.
Narok -> Bomet        [Clear]: Cruise safely and keep safe trailer spacing.
Bomet -> Kisii        [Clear]: Cruise safely and keep safe trailer spacing.
Kisii -> Kisumu       [Clear]: Cruise safely and keep safe trailer spacing.

Final route: Nairobi -> Narok -> Bomet -> Kisii -> Kisumu | Total: 400 km


### Step 8: Querying the Knowledge Base**

We query the KB directly for segment statuses and their driving rules, then run the agent through five scenarios (no hazards, blocked segments, a live report reopening a road, and all roads to Kisumu blocked) to show the route changing as the facts change.

In [9]:


# ---- Part A: direct queries (what does the KB say about a segment?) ----
print("=== Direct KB Queries ===")
queries = [
    ("Naivasha", "Nakuru"),      # general fact: Clear
    ("Londiani", "Muhoroni"),    # general fact: Slippery
    ("Nakuru", "Eldoret"),       # added in Step 5: Slippery
    ("Bomet", "Kericho"),        # added in Step 5: Blocked
    ("Thika", "Murang'a"),       # no fact stored -> defaults to Clear
]
for a, b in queries:
    status = get_status(a, b)
    print(f"{a} -> {b:12} [{status}]: {action_for(status)}")

print("Unknown condition 'Hailstorm':", action_for("Hailstorm"))

# ---- Part B: scenario queries (how does the agent plan under different facts?) ----
def run_scenario(title, observations):
    """Temporarily load a set of live observations, run the agent, then restore."""
    saved = dict(live_observations)
    live_observations.clear()
    live_observations.update(observations)

    print(f"\n\n##### {title} #####")
    plan_trip(start_city, goal_city)

    live_observations.clear()
    live_observations.update(saved)

run_scenario(
    "Scenario 1: No live hazards",
    {}
)

run_scenario(
    "Scenario 2: Mau Summit -> Londiani blocked",
    {("Mau Summit", "Londiani"): "Blocked"}
)

run_scenario(
    "Scenario 3: Londiani -> Muhoroni blocked (partial detour)",
    {("Londiani", "Muhoroni"): "Blocked"}
)

run_scenario(
    "Scenario 4: Live report reopens Bomet -> Kericho",
    {("Mau Summit", "Londiani"): "Blocked",
     ("Bomet", "Kericho"): "Clear"}
)

run_scenario(
    "Scenario 5: Every road into Kisumu blocked",
    {("Muhoroni", "Kisumu"): "Blocked",
     ("Kericho", "Kisumu"): "Blocked",
     ("Kisii", "Kisumu"): "Blocked",
     ("Kakamega", "Kisumu"): "Blocked"}
)

=== Direct KB Queries ===
Naivasha -> Nakuru       [Clear]: Cruise safely and keep safe trailer spacing.
Londiani -> Muhoroni     [Slippery]: Reduce speed early, brake gently, and prevent trailer sway.
Nakuru -> Eldoret      [Slippery]: Reduce speed early, brake gently, and prevent trailer sway.
Bomet -> Kericho      [Blocked]: Stop safely and reroute via Uniform Cost Search.
Thika -> Murang'a     [Clear]: Cruise safely and keep safe trailer spacing.
Unknown condition 'Hailstorm': Unknown condition - slow down and proceed with caution.


##### Scenario 1: No live hazards #####
TRAILER AGENT: Nairobi -> Kisumu
Original optimal route : Nairobi -> Naivasha -> Nakuru -> Mau Summit -> Londiani -> Muhoroni -> Kisumu (356 km)

No blocked segments on the route. Keeping original route.

--- Driving Instructions ---
Nairobi -> Naivasha     [Roadworks]: Slow down for diversions and watch for heavy machinery.
Naivasha -> Nakuru       [Clear]: Cruise safely and keep safe trailer spacing.
Nakuru -> 

### Step 9: Debugging the Knowledge Base**

We tested faulty inputs (unknown statuses, wrong towns, lowercase blocked, a missing start town). This exposed gaps and bugs, which we fixed by adding rules, making lookups case-insensitive, and validating towns.

In [10]:


# ---- Test 1: faulty or unknown road statuses ----
test_statuses = ["Clear", "Steep_Climb", "Blocked", "Hailstorm", "Flooded", "blocked", "good"]
print("\n--- Test 1: Statuses ---")
for s in test_statuses:
    print(f"{s:12} -> {action_for(s)}")

# ---- Test 2: faulty segments (typos, towns not in graph, no direct road) ----
def check_segment(a, b):
    a, b = clean(a), clean(b)
    if a not in highway_graph or b not in highway_graph:
        return "Unknown town"
    if b not in highway_graph[a] and a not in highway_graph[b]:
        return "No direct road in the graph"
    return f"OK [{get_status(a, b)}]"

print("\n--- Test 2: Segments ---")
for a, b in [("mau summit", "londiani"), ("Nairobi", "Kisii"), ("Nakuru", "Mombasa"), ("Murang'a", "Nyeri")]:
    print(f"{a} -> {b:10} : {check_segment(a, b)}")

# ---- Test 3: a lowercase 'blocked' should still block the road ----
print("\n--- Test 3: Case sensitivity ---")
live_observations.clear()
live_observations[("Mau Summit", "Londiani")] = "blocked"   # lowercase on purpose
print("Blocked segments found:", len(blocked_segments()), "(expected 4: two facts, both directions)")

# ---- Test 4: planning with a faulty start town ----
print("\n--- Test 4: Unknown start town ---")
try:
    plan_trip("Atlantis", "Kisumu")
except Exception as e:
    print("Agent crashed:", type(e).__name__, "-", e)

# ================= FIXES (edit the KB and the agent based on the tests) =================

# Fix 1: add rules for the statuses the agent didn't know
add_rule("Hailstorm", "Reduce speed, increase following distance and pull over if visibility drops.")
add_rule("Flooded", "Do not enter standing water; stop safely and reroute.")
add_rule("Good", "Maintain normal speed and keep safe trailer spacing.")

# Fix 2: make status lookups case-insensitive
def action_for(status):
    return trailer_rules.get(status.strip().title(), "Unknown condition - slow down and proceed with caution.")

def blocked_segments():
    blocked = set()
    for source in (segment_status, live_observations):
        for (a, b), status in source.items():
            if status.strip().title() in ("Blocked", "Flooded"):
                blocked.add((a, b)); blocked.add((b, a))
    for (a, b), status in live_observations.items():
        if status.strip().title() not in ("Blocked", "Flooded"):
            blocked.discard((a, b)); blocked.discard((b, a))
    return blocked

# Fix 3: remove the stray fact for a road that doesn't exist
segment_status.pop(("Nairobi", "Kisii"), None)

# Fix 4: validate towns before planning so the agent never crashes
def safe_plan_trip(start, goal):
    start, goal = clean(start), clean(goal)
    for town in (start, goal):
        if town not in highway_graph:
            print(f"Cannot plan: '{town}' is not in the road network.")
            return
    plan_trip(start, goal)

# ---- Re-run the tests after the fixes ----
print("\n\n===== After fixes =====")
for s in test_statuses:
    print(f"{s:12} -> {action_for(s)}")

print("\nBlocked segments found:", len(blocked_segments()))
safe_plan_trip("Atlantis", "Kisumu")

live_observations.clear()   # reset the live state for a clean run


--- Test 1: Statuses ---
Clear        -> Cruise safely and keep safe trailer spacing.
Steep_Climb  -> Use low gear and engine braking; avoid sudden steering.
Blocked      -> Stop safely and reroute via Uniform Cost Search.
Hailstorm    -> Unknown condition - slow down and proceed with caution.
Flooded      -> Unknown condition - slow down and proceed with caution.
blocked      -> Unknown condition - slow down and proceed with caution.
good         -> Maintain normal speed and keep safe trailer spacing

--- Test 2: Segments ---
mau summit -> londiani   : OK [Blocked]
Nairobi -> Kisii      : No direct road in the graph
Nakuru -> Mombasa    : Unknown town
Murang'a -> Nyeri      : OK [Clear]

--- Test 3: Case sensitivity ---
Blocked segments found: 2 (expected 4: two facts, both directions)

--- Test 4: Unknown start town ---
TRAILER AGENT: Atlantis -> Kisumu
Agent crashed: TypeError - can only join an iterable
Added rule: 'Hailstorm' -> 'Reduce speed, increase following distance and pull

### Step 10: Final Reflection**

**Challenges**
- Keeping general facts (Step 5) separate from live observations (Step 6), and deciding which wins when they conflict. We made live observations override stored facts.
- Debugging exposed that lowercase `blocked`, unknown statuses and unknown towns could make the agent drive into a closed road or crash.
- Roads are two-way, but our facts were stored in one direction, so the lookup had to check both.

**Recommended improvements**
- Pull live data from a traffic or weather API instead of typed input.
- Add a time stamp to observations so old reports expire.
- Use a proper rule engine or logic library instead of plain dictionaries.